# Diabetes Dataset — Data Preprocessing

This notebook performs preprocessing of the raw Pima Indians Diabetes dataset
for the Healytics multi-disease risk prediction system.

### Objectives

- Load the raw diabetes dataset
- Identify invalid zero-coded measurements
- Convert invalid values into missing values
- Separate features and target
- Split the dataset into training and testing sets
- Prepare the data for leakage-safe preprocessing and model development

> **Note:** This notebook does not perform model training.

## 1. Import Required Libraries

We use Pandas and NumPy for data loading and preprocessing operations.

In [1]:
import pandas as pd
import numpy as np

## 2. Load the Raw Dataset

The raw diabetes dataset is stored in the project's `data/raw` directory.

The dataset contains 768 records, 8 input features, and 1 target variable.

In [2]:
df = pd.read_csv("../data/raw/diabetes.csv")

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 3. Verify Dataset Shape

Before preprocessing, we verify that the dataset contains the expected
number of rows and columns.

In [5]:
df.shape

(768, 9)

## 4. Verify Column Names

The dataset contains eight input features and one target variable.

The target variable is `Outcome`, where:

- `0` represents no diabetes
- `1` represents diabetes

In [6]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='str')

## 5. Identify Invalid Zero Values

The raw Pima diabetes dataset uses zero values for some measurements where
zero is not physiologically meaningful.

The following features contain zero-coded missing measurements:

- Glucose
- BloodPressure
- SkinThickness
- Insulin
- BMI

However, zero is a valid value for:

- Pregnancies
- Outcome

In [15]:
zero_counts = (df == 0).sum()

zero_counts

Pregnancies                 111
Glucose                       0
BloodPressure                 0
SkinThickness                 0
Insulin                       0
BMI                           0
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64

## 6. Convert Invalid Zeros to Missing Values

The invalid zero values in medical measurements are converted to `NaN`.

We do not remove the corresponding rows because doing so would discard a
substantial amount of available data, particularly for the Insulin and
SkinThickness features.

The zero value in `Pregnancies` and `Outcome` is retained because it represents
a valid value.

In [7]:
invalid_zero_columns=["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df[invalid_zero_columns]=df[invalid_zero_columns].replace(0,np.nan)

## 7. Verify Missing Values After Conversion

After replacing invalid zeros with `NaN`, we check the number of missing
values in each column.

In [8]:
df.isnull().sum()

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

## 8. Separate Features and Target

The target variable is `Outcome`.

All remaining columns are used as input features.

- `X` → input features
- `y` → target variable

In [9]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [10]:
X.shape

(768, 8)

In [11]:
y.shape

(768,)

## 9. Split Data into Training and Testing Sets

The dataset is divided into training and testing sets.

- 80% of the data is used for training.
- 20% is reserved for final testing.
- `stratify=y` maintains a similar class distribution in both sets.
- `random_state=42` ensures reproducibility.

The test set must remain unseen during preprocessing and model training.

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (614, 8)
Testing features: (154, 8)
Training target: (614,)
Testing target: (154,)


## 10. Verify Class Distribution

Because the diabetes dataset has an imbalanced target distribution, we verify
the distribution of the target variable in both the training and testing sets.

Stratified splitting should maintain approximately the same proportion of
classes in both sets.

In [17]:
print("Training set:")
print(y_train.value_counts())
print()

print("Testing set:")
print(y_test.value_counts())

Training set:
Outcome
0    400
1    214
Name: count, dtype: int64

Testing set:
Outcome
0    100
1     54
Name: count, dtype: int64


In [18]:
print("Training distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print()

print("Testing distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training distribution (%):
Outcome
0    65.15
1    34.85
Name: proportion, dtype: float64

Testing distribution (%):
Outcome
0    64.94
1    35.06
Name: proportion, dtype: float64


## 11. Data Leakage Prevention

Missing-value imputation and other preprocessing operations must not use
information from the test set.

Therefore, preprocessing parameters such as median values will be learned
from the training data only.

The test data will only be transformed using parameters learned from the
training data.

This prevents information from the test set from influencing the model during
training.

## 12. Preprocessing Status

At this stage:

- Invalid zero-coded medical measurements have been identified.
- Invalid zeros have been converted to missing values.
- Features and target have been separated.
- The dataset has been split into training and testing sets.
- Class distribution has been checked.
- Data leakage prevention has been considered.

### Next Step

The next stage is to build a leakage-safe preprocessing pipeline for handling
missing values and, where required, feature scaling.

## 13. Missing-Value Imputation

The invalid zero-coded measurements were converted to missing values (`NaN`).

Since machine learning models generally cannot work directly with these missing
values, they need to be imputed.

Median imputation is used because it is less sensitive to extreme values than
mean imputation.

The imputation values are learned only from the training dataset and then
applied to both the training and testing datasets.

This approach prevents data leakage from the test set.

In [19]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

In [20]:
print("Missing values in X_train:", np.isnan(X_train_imputed).sum())
print("Missing values in X_test:", np.isnan(X_test_imputed).sum())

Missing values in X_train: 0
Missing values in X_test: 0


## 14. Inspect Imputation Values

The median values calculated from the training set are inspected to understand
how the missing measurements were replaced.

These values are learned only from the training data.

In [21]:
imputation_values = pd.Series(
    imputer.statistics_,
    index=X_train.columns
)

imputation_values

Pregnancies                   3.0000
Glucose                     117.0000
BloodPressure                72.0000
SkinThickness                29.0000
Insulin                     125.0000
BMI                          32.4000
DiabetesPedigreeFunction      0.3825
Age                          29.0000
dtype: float64

### Interpretation

The missing values in the training and testing datasets have been replaced
using the corresponding feature medians calculated from the training set.

## 15. Feature Scaling

The features in the diabetes dataset have different numerical ranges.

For example, `Glucose` and `Age` have different scales, while
`DiabetesPedigreeFunction` has a much smaller numerical range.

Standardization transforms the features so that they have approximately:

- Mean = 0
- Standard deviation = 1

Scaling is particularly useful for distance-based and gradient-based models
such as KNN, SVM, and Logistic Regression.

Tree-based models generally do not require feature scaling.

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [23]:
print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

Training shape: (614, 8)
Testing shape: (154, 8)


In [24]:
print("Training mean:")
print(X_train_scaled.mean(axis=0).round(2))

print("\nTraining standard deviation:")
print(X_train_scaled.std(axis=0).round(2))

Training mean:
[-0. -0.  0. -0. -0.  0. -0. -0.]

Training standard deviation:
[1. 1. 1. 1. 1. 1. 1. 1.]


## 16. Verify Scaled Test Data

The test dataset is transformed using the scaling parameters learned from the
training dataset.

The scaler is not fitted again on the test data, ensuring that information from
the test set does not influence preprocessing.

In [25]:
print("Testing mean:")
print(X_test_scaled.mean(axis=0).round(2))

print("\nTesting standard deviation:")
print(X_test_scaled.std(axis=0).round(2))

Testing mean:
[ 0.04 -0.    0.1   0.04  0.19  0.01 -0.08 -0.05]

Testing standard deviation:
[1.08 1.07 0.92 0.94 1.41 1.04 1.01 0.97]


## 17. Build a Reusable Preprocessing Pipeline

The preprocessing steps performed above will later need to be applied consistently
to unseen data and user inputs.

A Scikit-learn pipeline provides a reusable sequence of preprocessing operations.

The pipeline will:

1. Handle missing values using median imputation.
2. Standardize the numerical features.

The pipeline is fitted only on the training data.

In [26]:
from sklearn.pipeline import Pipeline

preprocessing_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [27]:
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

In [28]:
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (614, 8)
Processed testing shape: (154, 8)


In [29]:
print("Missing values in processed training data:",
      np.isnan(X_train_processed).sum())

print("Missing values in processed testing data:",
      np.isnan(X_test_processed).sum())

Missing values in processed training data: 0
Missing values in processed testing data: 0


## 18. Preprocessing Summary

The diabetes dataset has been successfully prepared for machine learning.

### Steps performed

1. Loaded the raw dataset.
2. Identified zero-coded missing measurements.
3. Converted invalid zeros to `NaN`.
4. Separated features and target.
5. Split the data using an 80/20 stratified train-test split.
6. Applied median imputation using training data only.
7. Standardized numerical features using training data only.
8. Created a reusable Scikit-learn preprocessing pipeline.
9. Verified that the processed datasets contain no missing values.

### Data Leakage Prevention

All preprocessing parameters were learned exclusively from the training
dataset. The test dataset was only transformed using the parameters learned
from the training dataset.

The processed data is now ready for the model development stage.

## 19. Pipeline Consistency Check

The reusable preprocessing pipeline is compared with the preprocessing steps
performed separately above.

The outputs should be equivalent up to numerical precision.

In [30]:
import numpy as np

print(
    "Training pipeline consistency:",
    np.allclose(X_train_scaled, X_train_processed)
)

print(
    "Testing pipeline consistency:",
    np.allclose(X_test_scaled, X_test_processed)
)

Training pipeline consistency: True
Testing pipeline consistency: True
